In [ ]:
# ============================================
# Build enriched book dataset
# Books.csv (Book-Crossing) + books_1.Best_Books_Ever.csv (Goodreads)
# Output: books_full_with_description.csv
# ============================================

import pandas as pd
import re

# ---------- 1. Load both datasets ----------
books = pd.read_csv('Books.csv', low_memory=False, dtype={'ISBN': str})
bbe   = pd.read_csv('books_1.Best_Books_Ever.csv', dtype={'isbn': str})

print("Book-Crossing rows:", len(books))
print("Best Books Ever rows:", len(bbe))

# ---------- 2. ISBN normalization ----------
# Book-Crossing uses 10-digit ISBNs; Best Books Ever mostly uses 13-digit.
# Convert ISBN-13 -> ISBN-10 so they can be matched.
def isbn13_to_isbn10(isbn13):
    isbn13 = str(isbn13).strip()
    if len(isbn13) != 13 or not isbn13.startswith('978'):
        return None
    core = isbn13[3:12]
    if not core.isdigit():
        return None
    total = sum((10 - i) * int(d) for i, d in enumerate(core))
    check = (11 - (total % 11)) % 11
    return core + ('X' if check == 10 else str(check))

def normalize_isbn(raw):
    raw = str(raw).strip().upper()
    if raw in ('9999999999999', '', 'NAN'):   # placeholder / missing ISBNs
        return None
    if len(raw) == 10:
        return raw
    if len(raw) == 13:
        return isbn13_to_isbn10(raw)
    return None

# ---------- 3. Title / author normalization (for fallback matching) ----------
def norm_title(t):
    t = str(t).lower()
    t = re.sub(r'[^a-z\s]', '', t)
    return ' '.join(t.split())

def norm_author(a):
    return re.sub(r'[^a-z]', '', str(a).lower())

bbe['isbn10']      = bbe['isbn'].apply(normalize_isbn)
bbe['title_key']   = bbe['title'].apply(norm_title)
bbe['author_key2'] = bbe['author'].apply(norm_author)

books['ISBN_norm']   = books['ISBN'].str.strip().str.upper()
books['title_key']   = books['Book-Title'].apply(norm_title)
books['author_key2'] = books['Book-Author'].apply(norm_author)

# ---------- 4. Match strategy 1: exact ISBN ----------
by_isbn = books.merge(
    bbe[['isbn10', 'description', 'genres']],
    left_on='ISBN_norm', right_on='isbn10', how='inner'
)
by_isbn['match_method'] = 'isbn'
print("ISBN matches:", len(by_isbn))

# ---------- 5. Match strategy 2: title + author (only for unmatched books) ----------
matched_isbns = set(by_isbn['ISBN'])
remaining = books[~books['ISBN'].isin(matched_isbns)]

by_title = remaining.merge(
    bbe[['title_key', 'author_key2', 'description', 'genres']],
    on=['title_key', 'author_key2'], how='inner'
).drop_duplicates(subset='ISBN')
by_title['match_method'] = 'title_author'
print("Additional title+author matches:", len(by_title))

# ---------- 6. Combine, drop rows with no description ----------
combined = pd.concat([by_isbn, by_title], ignore_index=True)
combined = (combined
            .dropna(subset=['description'])
            .drop_duplicates(subset='ISBN')
            .reset_index(drop=True))

print(f"Final: {len(combined)} books with descriptions "
      f"({len(combined)/len(books):.1%} of raw catalog)")

# ---------- 7. Save ----------
out_cols = ['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication',
            'Publisher', 'description', 'genres', 'match_method']
final = combined[out_cols].rename(columns={'genres': 'genres_raw'})
final.to_csv('books_full_with_description.csv', index=False)

print(final['match_method'].value_counts())
print(final.head())

Book-Crossing rows: 271360
Best Books Ever rows: 52478
ISBN matches: 8255
Additional title+author matches: 6756
Final: 14884 books with descriptions (5.5% of raw catalog)
match_method
isbn            8168
title_author    6716
Name: count, dtype: int64
         ISBN                                         Book-Title  \
0  0440234743                                      The Testament   
1  0609804618  Our Dumb Century: The Onion Presents 100 Years...   
2  0375759778                                   Prague : A Novel   
3  0553582747                         From the Corner of His Eye   
4  0440223571  This Year It Will Be Different: And Other Stories   

       Book-Author Year-Of-Publication                      Publisher  \
0     John Grisham                1999                           Dell   
1        The Onion                1999             Three Rivers Press   
2  ARTHUR PHILLIPS                2003  Random House Trade Paperbacks   
3      Dean Koontz                2001         